# Task 2 — Unsupervised Domain Adaptation on PACS (sources: Photo, Art Painting, Cartoon → target: Sketch)

Protocol (identical for every method, implemented once in `shared/pacs_engine.py`):
* ResNet-18 (`IMAGENET1K_V1`), 7-way linear head, whole network fine-tuned; BatchNorm running statistics frozen at ImageNet values (γ/β trainable).
* Resize 256×256 → random 224 crop + horizontal flip (train, and target images during adaptation); centre 224 crop for validation / evaluation.
* Every step: 8 labelled images from each source domain + 24 unlabelled Sketch images (methods that do not align ignore the target batch). One epoch = one pass over the largest source training split.
* AdamW (lr 1e-4, wd 1e-4), ≤ 30 epochs, early stopping after 5 epochs without improvement of the **mean source-validation macro-F1**, which also selects the checkpoint. Seed 6304 is reset before each run, so initialization and sampling are identical across methods.
* Transductive UDA: all Sketch images are used unlabelled during adaptation. **Sketch labels are read only in the "Final evaluation" section**, after every run (including the design study) has been trained and frozen.

Controlled design study (Step 6): **DAN with λ_MMD ∈ {0.1, 1, 10}** (λ = 1 is the main DAN run).

Set `QUICK_RUN = True` for a short dry run (2 epochs × 20 steps) — never report those numbers. Existing checkpoints in `task2/checkpoints/` are reused unless `RETRAIN = True`.

In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "common").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from common.seed import set_seed
from common.paths import results_dir, checkpoints_dir, figure_path
from common.logging import save_table, save_json
from shared.pacs import PACS_CLASSES, PACS_DOMAINS
from shared.pacs_engine import PACSTrainConfig, SOURCE_DOMAINS, load_history

SEED = 6304
QUICK_RUN = False
RETRAIN = False
NUM_WORKERS = 2            # per domain loader (4 loaders run concurrently)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS, CKPT = results_dir(2), checkpoints_dir(2)
set_seed(SEED)
pd.set_option("display.precision", 4); pd.set_option("display.width", 220)

cfg = PACSTrainConfig(seed=SEED, device=DEVICE, num_workers=NUM_WORKERS,
                      max_epochs=2 if QUICK_RUN else 30, max_steps_per_epoch=20 if QUICK_RUN else None)
print(cfg)

PACSTrainConfig(max_epochs=30, lr=0.0001, weight_decay=0.0001, patience=5, seed=6304, per_source_batch=8, target_batch=24, num_workers=2, eval_batch=128, device='cuda', max_steps_per_epoch=None)


## Data and the shared source splits
The stratified 80/20 split of every source domain (seed 6304) is written once to `shared/splits/pacs_sketch_seed6304.json` and reused verbatim by Task 3.

In [2]:
from task2.train import (get_splits, MAIN_RUNS, STUDY_RUNS, STUDY_LAMBDAS, study_run_name, train_run,
                         DANN_STUDY_RUNS, DANN_ALPHAS, dann_study_run_name)
splits = get_splits()
counts = []
for d in PACS_DOMAINS:
    lab = splits[d]["labels"]
    row = {"domain": d, "total": len(lab), "train": len(splits[d]["train_idx"]), "val": len(splits[d]["val_idx"])}
    row.update({c: int((lab == k).sum()) for k, c in enumerate(PACS_CLASSES)})
    counts.append(row)
display(save_table(counts, RESULTS / "dataset_counts"))

,domain,total,train,val,dog,elephant,giraffe,guitar,horse,house,person
0,photo,1670,1337,333,189,202,182,186,199,280,432
1,art_painting,2048,1638,410,379,255,285,184,201,295,449
2,cartoon,2344,1875,469,389,457,346,135,324,288,405
3,sketch,3929,3143,786,772,740,753,608,816,80,160


## Steps 1–4 — Source-only, DAN (λ_MMD = 1), DANN, CDAN
* **Source-only**: cross-entropy on domain-balanced source batches (this checkpoint is Task 3's ERM baseline).
* **DAN**: + λ·MMD²(pooled source, target) on the 512-d feature; 3 RBF kernels at 0.5/1/2 × median pairwise squared distance of the combined batch.
* **DANN**: + domain cross-entropy of a 512→256→2 discriminator (ReLU, dropout 0.5) behind a GRL with α(p) = 2/(1+e^{−10p}) − 1, p = progress over the full 30-epoch budget.
* **CDAN**: same discriminator/GRL/weight, input g = vec(f ⊗ softmax(head(f))) (3584-d), nothing detached, no entropy conditioning.

**Two implementation choices forced by the prescribed setup (both found from source-side training behaviour, before any target label was used):**
1. *MMD estimator.* The loss uses the **unbiased (U-statistic) MMD² estimate** (Gretton et al., 2012). The biased V-statistic contains the constant diagonal terms k(x,x) (≈ 3/n per sample for three kernels), which dominate at these batch sizes; together with the median bandwidth, which follows the feature scale, its gradient drives the features to an all-zero ReLU collapse (seen for DAN-DG in Task 3: loss stuck at ln 7). The same estimator is used by DAN-DG, as required.
2. *Discriminator input scale.* With BatchNorm statistics frozen and AdamW, nothing bounds the 512-d feature norm, and the reversed gradient inflates it to make the discriminator confidently wrong: an un-normalized DANN diverged within the first epoch (feature norm ×1.5 per step, losses ≈ 10⁵; log kept in `task2/results/diagnostics/`). The discriminator therefore receives the feature's direction at a **fixed norm of √512**, √512·f/‖f‖ (CDAN: vec(√512·f/‖f‖ ⊗ p)). Architecture, GRL schedule, unit loss weight and optimizer are unchanged; the discriminator can still use every direction of the feature, only not its norm.
   *Run v1 used the unit-norm f/‖f‖.* Its domain loss stayed near ln 2 (≈0.64 in the first epoch, GRL α < 0.17) and its accuracy then fell to chance, while a linear probe still separated source from Sketch at 99.6 %. `task2/diagnose_discriminator.py` isolates the input scale (frozen Source-only features, the prescribed discriminator and optimizer, no gradient reversal, no Sketch labels; `task2/results/diagnostics/discriminator_input_scale.json`). With unit-norm inputs the discriminator gets the sign of the domain right but its logits stay near zero (much higher held-out cross-entropy than at √512), and the normalization divides the reversed gradient reaching the features by ‖f‖ ≈ 30–55. The adversarial game was therefore played at near-zero logits with a weak gradient into the backbone. √512 keeps the protection against norm inflation at the raw feature's per-coordinate scale. DANN and CDAN were retrained with it; the v1 logs are kept in `task2/results/diagnostics/`.

In [3]:
models, summaries = {}, {}
for name in MAIN_RUNS:
    models[name], summaries[name] = train_run(name, splits, cfg, retrain=RETRAIN)
summary_rows = [{"run": n, "best_epoch": s["best_epoch"], "epochs_run": s["epochs_run"], "stopped_early": s["stopped_early"],
                 "best_mean_source_val_f1": s["best_mean_val_macro_f1"], "minutes": s.get("minutes")} for n, s in summaries.items()]
display(save_table(summary_rows, RESULTS / "training_summary_main"))

[source_only] loaded existing checkpoint /workspace/pa1-beyond-iid/task2/checkpoints/source_only.pth


[dan] loaded existing checkpoint /workspace/pa1-beyond-iid/task2/checkpoints/dan.pth


[dann] loaded existing checkpoint /workspace/pa1-beyond-iid/task2/checkpoints/dann.pth


[cdan] loaded existing checkpoint /workspace/pa1-beyond-iid/task2/checkpoints/cdan.pth


,run,best_epoch,epochs_run,stopped_early,best_mean_source_val_f1,minutes
0,source_only,10,16,True,0.9430,7.9068
1,dan,1,7,True,0.9234,6.0492
2,dann,5,11,True,0.9488,9.2492
3,cdan,8,14,True,0.9338,11.8302


## Step 6 — Controlled design study: DAN with λ_MMD ∈ {0.1, 1, 10}
Everything except λ is fixed. Expected effect of increasing alignment pressure (stated before looking at any target result): source validation performance should stay flat or fall slightly (the MMD term competes with the class loss), source-vs-target domain separability should fall, and target accuracy may rise at moderate λ but can fall at large λ if alignment starts mixing classes. Target results below are for analysis only; the main comparison stays at λ = 1 regardless.

In [4]:
for name in STUDY_RUNS:
    models[name], summaries[name] = train_run(name, splits, cfg, retrain=RETRAIN)
display(pd.DataFrame([{"run": n, "best_epoch": summaries[n]["best_epoch"], "epochs_run": summaries[n]["epochs_run"],
                       "best_mean_source_val_f1": summaries[n]["best_mean_val_macro_f1"]} for n in STUDY_RUNS]))

[dan_lambda0.1] loaded existing checkpoint /workspace/pa1-beyond-iid/task2/checkpoints/dan_lambda0.1.pth


[dan_lambda10] loaded existing checkpoint /workspace/pa1-beyond-iid/task2/checkpoints/dan_lambda10.pth


,run,best_epoch,epochs_run,best_mean_source_val_f1
0,dan_lambda0.1,5,11,0.9373
1,dan_lambda10,7,13,0.8937


### Supplementary study — DANN maximum gradient-reversal strength α_max ∈ {0.25, 0.5, 1}
The assignment's other design-study option, added **after** the main DANN run had been evaluated: in run v2 the main DANN (α_max = 1) collapsed on Sketch. Its training log shows the reason without any target label: after the first epoch the discriminator's accuracy stayed *below* chance with its loss above ln 2, i.e. the backbone kept flipping the domains faster than the discriminator could follow. This study asks how that depends on the adversarial strength. The three values are the assignment's, fixed in advance; the schedule shape and every other setting are unchanged; the main comparison stays at α_max = 1; and these target results are for analysis only.

In [5]:
for name in DANN_STUDY_RUNS:
    models[name], summaries[name] = train_run(name, splits, cfg, retrain=RETRAIN)
display(pd.DataFrame([{"run": n, "best_epoch": summaries[n]["best_epoch"], "epochs_run": summaries[n]["epochs_run"],
                       "best_mean_source_val_f1": summaries[n]["best_mean_val_macro_f1"]} for n in DANN_STUDY_RUNS]))

[dann_alpha0.25] loaded existing checkpoint /workspace/pa1-beyond-iid/task2/checkpoints/dann_alpha0.25.pth


[dann_alpha0.5] loaded existing checkpoint /workspace/pa1-beyond-iid/task2/checkpoints/dann_alpha0.5.pth


,run,best_epoch,epochs_run,best_mean_source_val_f1
0,dann_alpha0.25,7,13,0.9333
1,dann_alpha0.5,6,12,0.9442


## Training diagnostics (source data only)
Top row: training classification loss and mean source-validation macro-F1 (dotted line = selected epoch). Bottom row: the alignment term — MMD² for DAN; domain loss, discriminator accuracy (dotted = chance) and GRL α for DANN/CDAN — to check that each method trained as intended.

In [6]:
from shared.pacs_analysis import training_curves
histories = {n: load_history(RESULTS, n) for n in models}
training_curves({n: histories[n] for n in MAIN_RUNS}, figure_path("task2_training_curves.png"))
training_curves({study_run_name(l): histories[study_run_name(l)] for l in STUDY_LAMBDAS}, figure_path("task2_dan_lambda_training_curves.png"))
training_curves({dann_study_run_name(a): histories[dann_study_run_name(a)] for a in DANN_ALPHAS},
                figure_path("task2_dann_alpha_training_curves.png"))

PosixPath('/workspace/pa1-beyond-iid/report/figures/task2_dann_alpha_training_curves.png')

## Configuration lock
All checkpoints and settings are now fixed. The next section is the first point at which Sketch labels are used.

In [7]:
save_json({"config": {k: str(v) for k, v in cfg.__dict__.items()}, "runs": {n: summaries[n] for n in models},
           "checkpoints": {n: str(CKPT / f"{n}.pth") for n in models}}, RESULTS / "locked_configuration.json")

PosixPath('/workspace/pa1-beyond-iid/task2/results/locked_configuration.json')

## Step 5 — Final evaluation and alignment diagnostic
Domain separability: frozen 512-d features of all source-validation images and an equal number of (seeded) Sketch images, stratified 70/30 split (seed 6304), standardized features, class-balanced logistic regression with C = 1; held-out accuracy (50 % = chance).

In [8]:
from task2.evaluate_final import evaluate_models
from shared.pacs_analysis import source_rows
evals = evaluate_models(models, splits, cfg, seed=SEED)

def main_row(name):
    e = evals[name]
    r = {"run": name, **source_rows(e, list(SOURCE_DOMAINS))}
    t = e["sketch"]
    r["target_acc"] = float((t["preds"] == t["labels"]).mean())
    from common.metrics import macro_f1
    r["target_f1"] = macro_f1(t["labels"], t["preds"])
    r["domain_separability"] = e["domain_separability"]
    return r

rows = [main_row(n) for n in models]
base_acc = next(r["target_acc"] for r in rows if r["run"] == "source_only")
for r in rows:
    r["target_acc_change_vs_source_only"] = r["target_acc"] - base_acc
    r["source_to_target_gap_acc"] = r["mean_source_acc"] - r["target_acc"]
table = save_table([r for r in rows if r["run"] in MAIN_RUNS], RESULTS / "step5_main_comparison")
cols = ["run"] + [f"{d}_acc" for d in SOURCE_DOMAINS] + [f"{d}_f1" for d in SOURCE_DOMAINS] + \
       ["mean_source_acc", "mean_source_f1", "target_acc", "target_f1", "target_acc_change_vs_source_only", "domain_separability"]
display(table[cols])
np.savez_compressed(RESULTS / "target_predictions.npz", labels=evals["source_only"]["sketch"]["labels"],
                    paths=np.array(splits["sketch"]["paths"]),
                    **{f"{n}__preds": evals[n]["sketch"]["preds"] for n in models},
                    **{f"{n}__probs": evals[n]["sketch"]["probs"] for n in models})

,run,photo_acc,art_painting_acc,cartoon_acc,photo_f1,art_painting_f1,cartoon_f1,mean_source_acc,mean_source_f1,target_acc,target_f1,target_acc_change_vs_source_only,domain_separability
0,source_only,0.961,0.9341,0.9467,0.9515,0.9305,0.9470,0.9473,0.9430,0.5719,0.6020,0.0000,0.9973
1,dan,0.967,0.8927,0.9296,0.9594,0.8849,0.9260,0.9298,0.9234,0.7580,0.7322,0.1861,0.9299
2,dann,0.958,0.9488,0.9488,0.9496,0.9465,0.9503,0.9519,0.9488,0.2476,0.1329,-0.3243,0.9986
3,cdan,0.958,0.9098,0.9382,0.9512,0.9102,0.9400,0.9353,0.9338,0.6645,0.5806,0.0926,0.9918


In [9]:
# Lower separability vs. better target recognition?
from common.plotting import separability_vs_accuracy_scatter
separability_vs_accuracy_scatter(list(table.run), list(table.domain_separability), list(table.target_acc),
                                 "Task 2: domain separability vs. Sketch accuracy", figure_path("task2_separability_vs_accuracy.png"))
from scipy.stats import spearmanr
print(f"Spearman(separability, target acc) over all {len(rows)} runs:",
      spearmanr([r["domain_separability"] for r in rows], [r["target_acc"] for r in rows]))

Spearman(separability, target acc) over all 8 runs: SignificanceResult(statistic=np.float64(-0.8024095954818835), pvalue=np.float64(0.01654072280067719))


### Class-level analysis (target labels used only here)

In [10]:
from shared.pacs_analysis import (per_class_delta_rows, largest_change_rows, transition_counts, confusion_grid,
                                  per_class_delta_plot, domain_tsne, failure_grid)
tgt = {n: evals[n]["sketch"] for n in MAIN_RUNS}
pc = save_table(per_class_delta_rows(tgt, "source_only"), RESULTS / "step5_per_class_target")
display(pc.pivot(index="class", columns="run", values="accuracy"))
display(pc.pivot(index="class", columns="run", values="delta_vs_baseline"))
display(save_table(transition_counts(tgt, "source_only"), RESULTS / "step5_fixed_vs_broken"))
changes = largest_change_rows(tgt, "source_only")
save_json(changes, RESULTS / "step5_largest_changes_confusions.json")
for c in changes:
    print(f"{c['run']:6s} {c['kind']:14s} {c['class']:9s} delta={c['delta']:+.3f} | run confusions: "
          + ", ".join(f"{x['predicted_as']} {x['fraction_of_class']:.2f}" for x in c["confusions_run"])
          + " | source-only: " + ", ".join(f"{x['predicted_as']} {x['fraction_of_class']:.2f}" for x in c["confusions_baseline"]))
per_class_delta_plot(pc.to_dict("records"), "source_only", figure_path("task2_per_class_delta.png"),
                     ylabel="Sketch acc. change vs. Source-only")
confusion_grid(tgt, figure_path("task2_target_confusions.png"))
# Source-only's own domain gap per class (which classes drive the failures)
so_gap = []
for k, c in enumerate(PACS_CLASSES):
    src_acc = np.mean([(evals["source_only"][d]["preds"][evals["source_only"][d]["labels"] == k] == k).mean() for d in SOURCE_DOMAINS])
    t = evals["source_only"]["sketch"]
    so_gap.append({"class": c, "mean_source_val_acc": src_acc, "sketch_acc": (t["preds"][t["labels"] == k] == k).mean()})
display(save_table(so_gap, RESULTS / "step5_source_only_gap_per_class"))

run,cdan,dan,dann,source_only
class,,,,
dog,0.6826,0.5453,0.0816,0.4870
elephant,0.8676,0.6770,0.0027,0.7932
giraffe,0.5246,0.7955,0.9655,0.8088
guitar,0.8273,0.9704,0.2829,0.7204
horse,0.5453,0.7806,0.0110,0.1348
house,1.0000,1.0000,0.0000,0.9250
person,0.1187,0.9375,0.0000,0.3312


run,cdan,dan,dann,source_only
class,,,,
dog,0.1956,0.0583,-0.4054,0.0
elephant,0.0743,-0.1162,-0.7905,0.0
giraffe,-0.2842,-0.0133,0.1567,0.0
guitar,0.1069,0.2500,-0.4375,0.0
horse,0.4105,0.6458,-0.1238,0.0
house,0.0750,0.0750,-0.9250,0.0
person,-0.2125,0.6062,-0.3312,0.0


,run,fixed,broken,net,n
0,dan,1101,370,731,3929
1,dann,179,1453,-1274,3929
2,cdan,795,431,364,3929


dan    most_improved  horse     delta=+0.646 | run confusions: dog 0.05, house 0.05, giraffe 0.04 | source-only: giraffe 0.56, dog 0.21, elephant 0.09
dan    most_degraded  elephant  delta=-0.116 | run confusions: dog 0.19, person 0.08, house 0.02 | source-only: giraffe 0.10, dog 0.10
dann   most_improved  giraffe   delta=+0.157 | run confusions: dog 0.03, horse 0.00, elephant 0.00 | source-only: elephant 0.08, dog 0.08, horse 0.02
dann   most_degraded  house     delta=-0.925 | run confusions: giraffe 0.57, dog 0.40, guitar 0.01 | source-only: giraffe 0.06, elephant 0.01
cdan   most_improved  horse     delta=+0.411 | run confusions: dog 0.30, house 0.09, elephant 0.06 | source-only: giraffe 0.56, dog 0.21, elephant 0.09
cdan   most_degraded  giraffe   delta=-0.284 | run confusions: dog 0.29, horse 0.09, house 0.06 | source-only: elephant 0.08, dog 0.08, horse 0.02


,class,mean_source_val_acc,sketch_acc
0,dog,0.9215,0.4870
1,elephant,0.9786,0.7932
2,giraffe,0.9835,0.8088
3,guitar,0.9336,0.7204
4,horse,0.8808,0.1348
5,house,0.9885,0.9250
6,person,0.9255,0.3312


In [11]:
# Failure cases: Sketch images that adaptation fixed / broke relative to Source-only.
rng = np.random.RandomState(SEED)
paths_, titles_, fail_rows = [], [], []
base = tgt["source_only"]
spaths = np.array(splits["sketch"]["paths"])
for n in ["dan", "dann", "cdan"]:
    p = tgt[n]
    for kind, mask in [("broken", (base["preds"] == base["labels"]) & (p["preds"] != p["labels"])),
                       ("fixed", (base["preds"] != base["labels"]) & (p["preds"] == p["labels"]))]:
        for i in rng.permutation(np.where(mask)[0])[:4]:
            paths_.append(spaths[i])
            titles_.append(f"{n} {kind}\ntrue {PACS_CLASSES[p['labels'][i]]}\nSO:{PACS_CLASSES[base['preds'][i]]} -> {PACS_CLASSES[p['preds'][i]]}")
            fail_rows.append({"run": n, "kind": kind, "path": spaths[i], "true": PACS_CLASSES[p["labels"][i]],
                              "source_only_pred": PACS_CLASSES[base["preds"][i]], "run_pred": PACS_CLASSES[p["preds"][i]]})
save_table(fail_rows, RESULTS / "step5_failure_examples")
failure_grid(paths_, titles_, figure_path("task2_failure_examples.png"), n_cols=8)

PosixPath('/workspace/pa1-beyond-iid/report/figures/task2_failure_examples.png')

In [12]:
# t-SNE of source-validation and Sketch features (coloured by domain) for each method.
feats, doms = {}, {}
for n in MAIN_RUNS:
    feats[n] = np.concatenate([evals[n][d]["features"] for d in SOURCE_DOMAINS] + [evals[n]["sketch"]["features"]])
    doms[n] = sum([[d] * len(evals[n][d]["labels"]) for d in SOURCE_DOMAINS], []) + ["sketch"] * len(evals[n]["sketch"]["labels"])
domain_tsne(feats, doms, figure_path("task2_domain_tsne.png"), seed=SEED)

PosixPath('/workspace/pa1-beyond-iid/report/figures/task2_domain_tsne.png')

## Step 6 — Controlled study results

In [13]:
from shared.pacs_analysis import study_plot
study = [dict(r, lambda_mmd=l) for l in STUDY_LAMBDAS for r in rows if r["run"] == study_run_name(l)]
study_df = save_table(study, RESULTS / "step6_dan_lambda_study")
display(study_df[["lambda_mmd", "run", "photo_acc", "art_painting_acc", "cartoon_acc", "mean_source_acc", "mean_source_f1",
                  "domain_separability", "target_acc", "target_f1", "target_acc_change_vs_source_only"]])
study_plot(study_df, "lambda_mmd", {"Source validation": {"mean_source_acc": "mean acc", "mean_source_f1": "mean macro-F1"},
                                    "Domain separability": {"domain_separability": "probe accuracy"},
                                    "Sketch (target)": {"target_acc": "accuracy", "target_f1": "macro-F1"}},
           figure_path("task2_dan_lambda_study.png"), "λ_MMD")

,lambda_mmd,run,photo_acc,art_painting_acc,cartoon_acc,mean_source_acc,mean_source_f1,domain_separability,target_acc,target_f1,target_acc_change_vs_source_only
0,0.1,dan_lambda0.1,0.9580,0.9244,0.9424,0.9416,0.9373,0.9739,0.7216,0.6738,0.1497
1,1.0,dan,0.9670,0.8927,0.9296,0.9298,0.9234,0.9299,0.7580,0.7322,0.1861
2,10.0,dan_lambda10,0.9219,0.8780,0.8913,0.8971,0.8937,0.7995,0.6058,0.5672,0.0339


PosixPath('/workspace/pa1-beyond-iid/report/figures/task2_dan_lambda_study.png')

In [14]:
# Supplementary: DANN alpha_max study (analysis only; the main DANN row stays at alpha_max = 1).
dstudy = [dict(r, max_alpha=a) for a in DANN_ALPHAS for r in rows if r["run"] == dann_study_run_name(a)]
for r in dstudy:
    t = evals[r["run"]]["sketch"]
    counts = np.bincount(t["preds"], minlength=len(PACS_CLASSES)) / len(t["preds"])
    r["largest_predicted_class"], r["largest_predicted_frac"] = PACS_CLASSES[int(counts.argmax())], float(counts.max())
    h = histories.get(r["run"]) or load_history(RESULTS, r["run"])
    r["disc_acc_epoch0"], r["mean_disc_acc_after_epoch0"] = h[0]["disc_acc"], float(np.mean([x["disc_acc"] for x in h[1:]]))
dstudy_df = save_table(dstudy, RESULTS / "step6_dann_alpha_study")
display(dstudy_df[["max_alpha", "run", "mean_source_acc", "mean_source_f1", "domain_separability", "target_acc", "target_f1",
                   "target_acc_change_vs_source_only", "largest_predicted_class", "largest_predicted_frac",
                   "disc_acc_epoch0", "mean_disc_acc_after_epoch0"]])
study_plot(dstudy_df, "max_alpha", {"Source validation": {"mean_source_acc": "mean acc", "mean_source_f1": "mean macro-F1"},
                                    "Domain separability": {"domain_separability": "probe accuracy"},
                                    "Sketch (target)": {"target_acc": "accuracy", "target_f1": "macro-F1"}},
           figure_path("task2_dann_alpha_study.png"), "DANN max α")

,max_alpha,run,mean_source_acc,mean_source_f1,domain_separability,target_acc,target_f1,target_acc_change_vs_source_only,largest_predicted_class,largest_predicted_frac,disc_acc_epoch0,mean_disc_acc_after_epoch0
0,0.25,dann_alpha0.25,0.9373,0.9333,0.9890,0.5747,0.6162,0.0028,dog,0.4864,0.8634,0.5648
1,0.50,dann_alpha0.5,0.9468,0.9442,0.9986,0.2807,0.1780,-0.2912,elephant,0.8992,0.7211,0.4881
2,1.00,dann,0.9519,0.9488,0.9986,0.2476,0.1329,-0.3243,giraffe,0.8531,0.6075,0.4026


PosixPath('/workspace/pa1-beyond-iid/report/figures/task2_dann_alpha_study.png')

## Research questions (answer in your own words)
1. Size of the Source-only domain gap and the classes behind the most important failures → `step5_main_comparison`, `step5_source_only_gap_per_class`, `task2_target_confusions`.
2. Does lower separability correspond to better target recognition? → `task2_separability_vs_accuracy`, `step5_per_class_target`, `step5_fixed_vs_broken`.
3. Class-conditional (CDAN) vs. marginal (DAN, DANN) alignment → per-class deltas, confusions, `task2_domain_tsne`.
4. Alignment-strength trade-off, and which λ could have been chosen without target labels (e.g. from source-validation F1 alone) → `step6_dan_lambda_study`.